# Efficient Drug Discovery using Molecular Data
## Notebook 1: Exploratory Data Analysis (EDA)
**Author:** Dev Kapania | IIT Roorkee Research Intern

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline
print('Libraries loaded!')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/drug_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('Dataset Info:')
print(df.info())
print('\nBasic Statistics:')
df.describe()

## 2. Missing Values & Duplicates

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

## 3. Target Distribution (Active vs Inactive)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['activity'].value_counts().plot(kind='bar', ax=axes[0], color=['#8e44ad', '#2980b9'])
axes[0].set_title('Drug Activity Distribution')
axes[0].set_xlabel('Activity (0=Inactive, 1=Active)')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

df['activity'].value_counts().plot(kind='pie', ax=axes[1],
    labels=['Inactive', 'Active'],
    colors=['#2980b9', '#8e44ad'],
    autopct='%1.1f%%')
axes[1].set_title('Active vs Inactive Proportion')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../data/processed/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Class counts:\n{df["activity"].value_counts()}')

## 4. Molecular Descriptor Distributions

In [ ]:
# Plot first 9 numerical features
num_cols = df.select_dtypes(include=np.number).drop(columns=['activity']).columns[:9]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[df['activity']==0][col], alpha=0.6, label='Inactive', color='#2980b9', bins=25)
    axes[i].hist(df[df['activity']==1][col], alpha=0.6, label='Active',   color='#8e44ad', bins=25)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Molecular Descriptor Distributions by Activity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/descriptor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
num_df = df.select_dtypes(include=np.number)
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.3)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Top Features Correlated with Activity

In [ ]:
corr_with_target = num_df.corr()['activity'].drop('activity').abs().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
corr_with_target.head(15).plot(kind='bar', color='#8e44ad')
plt.title('Top 15 Features Correlated with Drug Activity', fontsize=12, fontweight='bold')
plt.ylabel('Absolute Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/top_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most important features:')
print(corr_with_target.head(10))